In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 固定 100 对原因诊断（待单独执行授权）
仅诊断：100 对 × 两条件 × 正负两侧 = 400 条记录，science_denominator=0。
不限制 GPU 型号。不会生成图片、校准阈值或修改方法。
本 notebook 尚未执行；代码提交需先按授权发布。Secrets：HF_TOKEN、CEG_WM_ROOT_KEY。


In [ ]:
from pathlib import Path
from google.colab import auth, userdata
import json, os, subprocess, sys
EXACT = '25c38934c635fc8891efe7115cada8eb29f7c0ef'
REPO = Path('/content/ceg-wm-rotation-diagnostic-25c3893')
INPUT = Path('/content/rotation-diagnostic-input')
ASSETS = Path('/content/rotation-diagnostic-runtime')
OUTPUT = Path('/content/drive/MyDrive/CEG-WM/RotationFailure-Diagnostic-V1/diagnostic-v1')
PREFLIGHT = OUTPUT.parent / 'preflight-v1'
if (OUTPUT/'summary.json').exists():
    raise FileExistsError('Completed diagnostic retained; do not rerun.')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','RotationFailure-Diagnostic-V1','--single-branch',
                    'https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('Existing checkout has changes; do not overwrite.')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==EXACT
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip()
sys.path.insert(0,str(REPO/'diagnostics/rotation_failure_v1'))
from preflight import assess
from diagnostic import CONDITIONS
summary=json.loads((PREFLIGHT/'result.json').read_text())
assert summary['status']=='PREFLIGHT_PASSED' and summary['science_denominator']==0
for condition in CONDITIONS:
    row=json.loads((PREFLIGHT/(condition+'.json')).read_text())
    assert row['execution_kind']=='FROZEN_REAL_RUNTIME' and not assess(row)
print('Prior preflight records checked; no model loaded.')


## 读取原始输入
只读 Drive API；图片缓存留在 Colab 本地。读取结束后核验全部 200 张图及 400 条原始来源记录。


In [ ]:
auth.authenticate_user()
subprocess.run([sys.executable,str(REPO/'diagnostics/rotation_failure_v1/prepare_inputs.py'),
                '--root',str(INPUT)],check=True)
for name in ('HF_TOKEN','CEG_WM_ROOT_KEY'):
    try:
        value=userdata.get(name)
    except Exception:
        raise RuntimeError(f'Enable notebook access to Colab Secret {name}') from None
    if not value:
        raise RuntimeError(f'Empty secret: {name}')
    os.environ[name]=value
del value


## 执行完整诊断（仅在另行授权后运行）
本单元会加载真实模型并计算固定 400 条记录。中断后保留同一输出目录；所有已写行包括失败行均跳过，不自动重试。


In [ ]:
env=os.environ.copy()
env['PYTHONDONTWRITEBYTECODE']='1'
process=subprocess.run([sys.executable,str(REPO/'diagnostics/rotation_failure_v1/diagnostic.py'),
    '--root',str(INPUT),'run-diagnostic','--runtime-root',str(ASSETS),
    '--output',str(OUTPUT),'--execute-authorized-diagnostic'],cwd=REPO,env=env)
print('Runner exit:',process.returncode)
if process.returncode:
    print('Stopped: retain all files. Review the coverage cell before any continuation.')


## 只读检查结果覆盖
也可在中断后运行。失败和缺失不会从分母中删除；本表不作原因裁决或方法通过结论。


In [ ]:
subprocess.run([sys.executable,str(REPO/'diagnostics/rotation_failure_v1/check_results.py'),
                '--output',str(OUTPUT)],check=True)
if (OUTPUT/'summary.json').exists():
    print((OUTPUT/'summary.json').read_text())
else:
    print('No terminal summary: diagnostic is unfinished.')
